<a href="https://colab.research.google.com/github/clifcovickh/BiLSTMvsBiLSTMSentiment/blob/main/4_Dataset_Recovery_Tool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install feedparser
!pip install deep_translator

import pandas as pd
import numpy as np
import feedparser
import yfinance as yf
import torch
import urllib.parse
from datetime import datetime, timedelta
from transformers import pipeline
from deep_translator import GoogleTranslator
import warnings
import time

warnings.filterwarnings('ignore')
print("Initializing AI Models (FinBERT & Google Translator)...")
finbert = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=0 if torch.cuda.is_available() else -1)
translator = GoogleTranslator(source='id', target='en')

In [ ]:
MASTER_FILE_INPUT = 'BBCA_Master_Dataset_BiLSTM.csv'
master_df = pd.read_csv(MASTER_FILE_INPUT)

master_df['Date'] = pd.to_datetime(master_df['Date'], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d')

# Identify the gap
last_date_str = master_df['Date'].dropna().max()
last_date_obj = datetime.strptime(last_date_str, '%Y-%m-%d')
today_obj = datetime.now()
days_diff = (today_obj - last_date_obj).days

if days_diff <= 0:
    print(f"our dataset is already up to date (Last date: {last_date_str}). No recovery needed.")
else:
    print(f"Gap detected: Last update was {last_date_str} ({days_diff} days ago).")
    search_range = f"{days_diff + 2}d"

In [ ]:
search_keyword = "saham BBCA"
encoded_search_keyword = urllib.parse.quote_plus(search_keyword)
rss_url = f"https://news.google.com/rss/search?q={encoded_search_keyword}+when:{search_range}&hl=id&gl=ID&ceid=ID:id"

print(f"🔍 Testing Scraper Target: {rss_url}")
feed = feedparser.parse(rss_url)

test_rows = []
print("Running text strings through Translation & FinBERT Pipeline...")

for i, entry in enumerate(feed.entries[:100]):
    pub_date = pd.to_datetime(entry.published).strftime('%Y-%m-%d')
    headline = entry.title

    try:
        time.sleep(0.3) # Rate limit safeguard for translation server

        english_text = translator.translate(headline)

        pipeline_output = finbert(english_text)[0]
        assigned_label = pipeline_output['label'].capitalize() # Matches "Positive", "Negative", "Neutral"
        confidence_score = pipeline_output['score']

        # 3. Convert categorical output into your custom scaled continuous score (-1 to +1)
        if assigned_label == "Positive":
            sentiment_score = confidence_score
        elif assigned_label == "Negative":
            sentiment_score = -confidence_score
        else:
            sentiment_score = 0.0

    except Exception as e:
        print(f"Row {i} failed parsing. Error type: {str(e)}")
        english_text = f"[ERROR: {str(e)[:30]}]"
        sentiment_score = 0.0
        assigned_label = "Neutral"

    test_rows.append({
        'News_Date': pub_date,
        'Raw_Headline_ID': headline,
        'Translated_Headline_EN': english_text,
        'FinBERT_Label': assigned_label,
        'Sentiment_Score': round(sentiment_score, 4)
    })

df_test_inspection = pd.DataFrame(test_rows)

print("\n📊 NUMBER OF HEADLINES CAPTURED PER DATE:")
print("-" * 55)
print(df_test_inspection['News_Date'].value_counts().sort_index().to_string())
print("-" * 55)

inspection_path = '/content/BBCA_Scraped_Headlines_Inspection.csv'
df_test_inspection.sort_values(by=['News_Date', 'Sentiment_Score'], ascending=[False, False]).to_csv(inspection_path, index=False)

display(df_test_inspection[['News_Date', 'Raw_Headline_ID', 'FinBERT_Label', 'Sentiment_Score']].sample(min(3, len(df_test_inspection))))

In [ ]:
print(f"Step 2: Downloading missing stock prices since {last_date_str}...")

bbca_data = yf.download("BBCA.JK", start=last_date_str, progress=False)

if isinstance(bbca_data.columns, pd.MultiIndex):
    bbca_data.columns = bbca_data.columns.get_level_values(0)

bbca_data = bbca_data.reset_index()
bbca_data['Date'] = bbca_data['Date'].dt.strftime('%Y-%m-%d')

price_cols = ['Open', 'High', 'Low', 'Close']
for col in price_cols:
    bbca_data[col] = pd.to_numeric(bbca_data[col], errors='coerce').round(0).astype(int)

bbca_data['Volume'] = pd.to_numeric(bbca_data['Volume'], errors='coerce').astype(int)

# Sentiment of (Date - 1 Day) mapped to (Date)
df_daily_sentiment['Date_Mapped'] = (pd.to_datetime(df_daily_sentiment['News_Date']) + pd.Timedelta(days=1)).dt.strftime('%Y-%m-%d')

# Join Stock Date with the Mapped Sentiment Date
recovery_block = pd.merge(
    bbca_data[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']],
    df_daily_sentiment[['Date_Mapped', 'Sentiment_Score']],
    left_on='Date',
    right_on='Date_Mapped',
    how='left'
)

recovery_block = recovery_block.drop(columns=['Date_Mapped'])
recovery_block['Sentiment_Score'] = recovery_block['Sentiment_Score'].fillna(0)

print(f"Aligned {len(recovery_block)} trading days. Formatting is now standardized.")

In [ ]:
print("Step 3: Updating the Master Database...")

master_df = master_df[~master_df['Date'].isin(recovery_block['Date'])]
master_df = pd.concat([master_df, recovery_block], ignore_index=True)

master_df = master_df.sort_values('Date').reset_index(drop=True)
master_df.to_csv('BBCA_Master_Dataset_Lagged.csv', index=False)

print("="*50)
print(f"DATABASE RESTORED: Current as of {master_df['Date'].max()}")
print("="*50)
display(master_df.tail(5))